In [18]:
import pandas as pd
import sys
sys.path.append("..")
from src.stats import wilson_interval, two_prop_z
df = pd.read_csv("../data/clean/upworthy-archive-exploratory-packages-03.12.2020 (1).csv")
clean = df[df["problem"]==0].copy()
clean["ctr"] = clean["clicks"] / clean["impressions"]
print(clean.shape)

(18388, 19)


In [2]:
#best and runner up per test
ranked = clean.sort_values(["clickability_test_id", "ctr"], ascending = [True, False], kind = "stable")
top2 = ranked.groupby("clickability_test_id").head(2)
gap = top2.groupby("clickability_test_id")["ctr"].agg(lambda s: s.iloc[0] - s.iloc[1])


print(len(gap))
print((gap * 100).describe())
print(f"tests where best and runner up tie: {(gap == 0).sum()}")

3823
count    3823.000000
mean        0.274010
std         0.307922
min         0.000000
25%         0.075295
50%         0.178992
75%         0.358968
max         4.573941
Name: ctr, dtype: float64
tests where best and runner up tie: 2


In [3]:
ties = gap[gap == 0].index
print(clean[clean["clickability_test_id"].isin(ties)][["clickability_test_id", "impressions", "clicks"]])

           clickability_test_id  impressions  clicks
5757   5329c6bc5696642620000049         2079      47
5758   5329c6bc5696642620000049         2020      50
5759   5329c6bc5696642620000049         1968      36
12164  53507921f6bdf72973000006           24       0
13137  53507921f6bdf72973000006           21       0
13204  53507921f6bdf72973000006           13       0
13364  53507921f6bdf72973000006           28       0
21865  5329c6bc5696642620000049         2020      50


In [6]:
# Per Test Table
# For each test, we want to to see what are the best and runner up packages, and what is the lift/gap?

ranked["rank"] = ranked.groupby("clickability_test_id").cumcount() # cumcount() ranks the rows within each group, starting from 0. So the best package will have rank 0, the runner up will have rank 1, etc.
cols = ["clicks", "impressions", "ctr"]
best = ranked[ranked["rank"] == 0].set_index("clickability_test_id")[cols]
runner = ranked[ranked["rank"] == 1].set_index("clickability_test_id")[cols]

#one row per test: bets on the left, runner up on the right
table = best.join(runner, lsuffix = "_best", rsuffix="_runner")
# number of packages in each test
table["n_packages"] = clean.groupby("clickability_test_id").size()

#lift = best ctr - runner up ctr
table["lift"] = table["ctr_best"] - table["ctr_runner"]
# tie = no winner
table["tie"] = table["lift"] == 0

print(table.shape)
print((table["lift"] - gap).abs().max())
print(table["tie"].sum())

(3823, 9)
0.0
2


In [8]:
ranked.head(10)

,created_at,updated_at,clickability_test_id,excerpt,headline,lede,slug,eyecatcher_id,impressions,clicks,significance,first_place,winner,share_text,square,test_week,test_start,problem,ctr,rank
5942,2013-02-24 07:08:08+00:00,2016-04-02 16:26:25.064,51436061220cb800020001e7,"Good show, father.",The One Where A Creationist Picks A Fight And ...,"<p>In a debate about creationism, one guy got ...",the-one-where-a-creationist-picks-a-fight-and-...,5332ba141fae79f09f002c4f,2629,68,100.0,True,True,NaN,A_Priest_Ridicules_Creationist_-_YouTube.jpg,201308,2013-02-24 07:07:13+00:00,0,0.025865,0
6136,2013-02-24 07:09:18+00:00,2016-04-02 16:26:28.804,51436061220cb800020001e7,"Well played, God. Well played.",God Finds Out About Creationism And Sends A Re...,"<p>In a debate about creationism, one guy got ...",god-finds-out-about-creationism-and-sends-a-re...,5332ba141fae79f09f002c4f,2661,63,61.4,False,False,NaN,A_Priest_Ridicules_Creationist_-_YouTube.jpg,201308,2013-02-24 07:07:13+00:00,0,0.023675,1
6135,2013-02-24 07:07:13+00:00,2016-04-02 16:26:28.801,51436061220cb800020001e7,Why is he wearing that costume?,Creationism Shouldn't Be Taught In Science Cla...,"<p>In a debate about creationism, one guy got ...",creationism-shouldnt-be-taught-in-science-clas...,5332ba141fae79f09f002c4f,2539,49,11.9,False,False,NaN,A_Priest_Ridicules_Creationist_-_YouTube.jpg,201308,2013-02-24 07:07:13+00:00,0,0.019299,2
785,2013-02-24 07:10:53+00:00,2016-04-02 16:24:07.081,51436061220cb800020001e7,Religion and science can indeed be friends.,Creationism Has Nothing To Do With Christianit...,"<p>In a debate about creationism, one guy got ...",creationism-has-nothing-to-do-with-christianit...,5332ba141fae79f09f002c4f,2551,39,0.8,False,False,NaN,A_Priest_Ridicules_Creationist_-_YouTube.jpg,201308,2013-02-24 07:07:13+00:00,0,0.015288,3
6009,2013-03-17 07:36:27+00:00,2016-04-02 16:26:26.456,51436069220cb800020005ae,It's about time we had some truth in advertising.,"The Advertisements You Read Every Day, Only Naked",Learn what advertisements say to each other be...,the-advertisements-you-read-every-day-only-nak...,5332b5961fae79f09f000636,1986,44,100.0,True,True,NaN,Screenshot_3_17_13_1_32_AM_MDT.jpg,201311,2013-03-17 07:36:27+00:00,0,0.022155,0
6010,2013-03-17 07:37:01+00:00,2016-04-02 16:26:26.459,51436069220cb800020005ae,"Finally, some honesty.",Wouldn't It Be Great If Ads Told The Truth?,"Refreshing, horrifying honesty.",wouldnt-it-be-great-if-ads-told-the-truth,5332b5961fae79f09f000636,2003,21,1.1,False,False,NaN,Screenshot_3_17_13_1_32_AM_MDT.jpg,201311,2013-03-17 07:36:27+00:00,0,0.010484,1
6012,2013-03-17 07:42:10+00:00,2016-04-02 16:26:26.464,51436069220cb800020005ae,"It's about time, ads!",What If Advertisements Just Said What They Mea...,"Nice to have a little honesty for once, no?\n",what-if-advertisements-just-said-what-they-meant,5332b5961fae79f09f000636,1951,16,0.0,False,False,NaN,Screenshot_3_17_13_1_32_AM_MDT.jpg,201311,2013-03-17 07:36:27+00:00,0,0.008201,2
6011,2013-03-17 07:37:25+00:00,2016-04-02 16:26:26.462,51436069220cb800020005ae,Rude.,What Your Advertisement Say About You Behind C...,"Nice to have a little honesty for once, no?",what-your-advertisement-say-about-you-behind-c...,5332b5961fae79f09f000636,1907,15,0.1,False,False,NaN,Screenshot_3_17_13_1_32_AM_MDT.jpg,201311,2013-03-17 07:36:27+00:00,0,0.007866,3
800,2013-03-17 07:38:21+00:00,2016-04-02 16:24:11.525,51436069220cb800020005ae,"Honesty is so refreshing. Also, rude.",The One Where Your Advertisements Talk To You ...,Learn what advertisements say to each other be...,the-one-where-your-advertisements-talk-to-you-...,5332b5961fae79f09f000636,1950,15,0.0,False,False,NaN,Screenshot_3_17_13_1_32_AM_MDT.jpg,201311,2013-03-17 07:36:27+00:00,0,0.007692,4
5940,2013-02-07 05:42:25+00:00,2016-04-02 16:26:25.052,51436069220cb800020005bd,You have so much potential. Why are you wastin...,"Mr. President, I'm Not Mad. I'm Just Disappoin...","<p>The President made a lot of promises, and h...",mr-preside

In [16]:
table. describe()

,clicks_best,impressions_best,ctr_best,clicks_runner,impressions_runner,ctr_runner,n_packages,lift
count,3823.000000,3823.000000,3823.000000,3823.000000,3823.000000,3823.000000,3823.000000,3823.000000
mean,60.235940,3497.970180,0.017833,50.930421,3494.770599,0.015093,4.809835,0.002740
std,43.798027,1321.183984,0.011136,37.069001,1270.419108,0.010014,1.240005,0.003079
min,0.000000,24.000000,0.000000,0.000000,21.000000,0.000000,2.000000,0.000000
25%,33.000000,2767.500000,0.010100,27.000000,2779.500000,0.008277,4.000000,0.000753
50%,50.000000,3121.000000,0.015196,42.000000,3118.000000,0.012764,5.000000,0.001790
75%,77.000000,4025.500000,0.022635,65.000000,4028.500000,0.019173,6.000000,0.003590
max,975.000000,25314.000000,0.122842,702.000000,14970.000000,0.109397,14.000000,0.045739


In [ ]:
## P-values and intervals
# Our question is: for each test, could the best package's lead be luck & what range is each package's true click rate in?


In [22]:
# p-value: if the two packages were equally good, how often would the best lead by this much?
# the function takes whole column and one call covers all the 3623 tests

z, p = two_prop_z(table["clicks_best"], table["impressions_best"],
                 table["clicks_runner"], table["impressions_runner"])

table["z"] = z
table["p_value"] = p

#wilson range for each packages own ctr
table["best_low"], table["best_high"] = wilson_interval(table["clicks_best"], table["impressions_best"])
table["runner_low"], table["runner_high"] = wilson_interval(table["clicks_runner"], table["impressions_runner"])

print(f"tests with no p-value: {table['p_value'].isna().sum()}")
print(f"tests with p < 0.05: {(table['p_value']<0.05).sum()} of {len(table)}")

tests with no p-value: 1
tests with p < 0.05: 382 of 3823
